# 02 — Data Cleaning Pipeline

In this notebook, we're going to clean up the NYC Airbnb data based on our findings from the extraction phase. 

**Our Cleaning Checklist:**
1. **Fix column names**: Make them easier to work with (snake_case).
2. **Handle missing values**: Fill in blanks for reviews and names.
3. **Fix data types**: Ensure dates are actually dates.
4. **Handle outliers**: Remove $0 prices and cap extreme values.
5. **Feature engineering**: Add some useful columns like `price_tier` and `revenue_proxy` for our dashboard.

### Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path so we can import our cleaning script
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from scripts.etl_pipeline import (
    normalize_columns, rename_verbose_columns, drop_duplicates,
    convert_types, fill_missing, treat_price_outliers, 
    treat_min_nights, engineer_features, FINAL_COLUMNS
)

raw_path = project_root / 'data' / 'raw' / 'AB_NYC_2019.csv'
processed_path = project_root / 'data' / 'processed' / 'airbnb_nyc_cleaned.csv'

### Loading and Standardizing
First, we'll load the data and fix those long column names.

In [ ]:
df = pd.read_csv(raw_path)
print(f"Initial shape: {df.shape}")

df = normalize_columns(df)
df = rename_verbose_columns(df)
df.head(2)

### Cleaning Phase
Now we'll run through our cleaning steps: removing duplicates, fixing types, and handling missing data.

In [ ]:
df = drop_duplicates(df)
df = convert_types(df)
df = fill_missing(df)

print(f"Shape after basic cleaning: {df.shape}")

### Fixing Outliers
We'll remove the $0 listings and cap the price at the 99th percentile to avoid skewing our analysis.

In [ ]:
df = treat_price_outliers(df)
df = treat_min_nights(df)

print(f"Final price range: ${df['price'].min()} to ${df['price'].max()}")

### Feature Engineering
Let's add some extra columns that will make our analysis easier later on.

In [ ]:
df = engineer_features(df)
df[['price', 'price_tier', 'revenue_proxy', 'occupancy_rate_est']].head()

### Save the Results
Finally, we'll select our core columns and save the cleaned dataset.

In [ ]:
df_final = df[[c for c in FINAL_COLUMNS if c in df.columns]].copy()
df_final.to_csv(processed_path, index=False)

print(f"Success! Cleaned data saved to {processed_path}")
print(f"Final listing count: {len(df_final):,}")